# Topic 06 · VQ-VAE: turning images into tokens: a VQ-VAE built on Fashion-MNIST, from the codebook to new images

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd06-vq-vae-en.ipynb)

This notebook accompanies [Topic 06 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vq-vae/), a deep-learning course originally in Vietnamese.
It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/cd06-vq-vae.ipynb)): the code,
the data and every check are the same.

Lesson 19 compresses an image into a few *real* numbers. The language models of Lessons 14 and 17, on the other hand,
only read and write **tokens**: integers from a vocabulary. **VQ-VAE**
([van den Oord, Vinyals & Kavukcuoglu, 2017](https://arxiv.org/abs/1711.00937)) connects the two worlds: the encoder
turns an image into a grid of vectors, each vector is replaced by the index of the nearest vector in a **codebook**,
and the decoder rebuilds the image from that grid of indices. The image becomes a sequence of tokens, so a
Transformer like Lesson 17's can learn to write new images, token by token. VQGAN, MaskGIT, Parti, Chameleon and the
audio tokenizers of Topic 05 are all built on this idea.

We build a VQ-VAE from scratch on Fashion-MNIST, at a size that runs on a CPU, test claims from the paper and from
later work on 3 seeds, then generate new clothes from tokens.

**What you'll do:**
1. Count bits: a 28 × 28 image becomes 49 tokens of 7 bits each.
2. Write the quantization step, the *straight-through* trick and the three loss terms yourself, then ask autograd
   which term trains which part of the network.
3. See the codebook "die" (*codebook collapse*) right at initialization, and fix it.
4. Train a VQ-VAE live, then look at what each token carries.
5. Rebuild the "one token for the whole image" example of Kyutai's article: clustering on top, quantizing during
   training, and two-level residual quantization (RVQ).
6. Test six claims about VQ-VAE with results on 3 seeds.
7. Train a small Transformer prior on the tokens, then generate and complete images.

**Background:** [Lesson 11 · Convolutional networks (CNNs)](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/cnn/),
[Lesson 14 · Tokens & embeddings](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-embedding/),
[Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/) and
[Lesson 19 · Autoencoders & VAEs](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/autoencoder-vae/).
**Run time:** about a minute on a CPU, including the first download of Fashion-MNIST (~30 MB).

The notebook uses **PyTorch**, like the course's other topics. The live training runs the first 400 steps of the
reference model (seed 0) and the first 1,000 steps of the "one token for the whole image" autoencoder of section 8;
the full numbers (3 seeds, 22 configurations and the long runs) were computed in advance by the course's script
`reel/vqvae/sweep.py`, with the same code and the same seeds, and are loaded in later sections.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

`torch.use_deterministic_algorithms(True)` makes PyTorch use operations that give the same result on every run.
Without it, the codebook's gradient is summed by several threads in an order that changes from run to run, and two
training runs with the same seed already differ after a few dozen steps.

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import math
import os
import urllib.request
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def num(x, digits=2):
    """A number with thousands separators: 13,007 · 0.1 · −4.38 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=1):
    return num(100 * p, digits) + "%"


def sci(x, digits=1):
    """A very small or very large number: 3.2·10⁻⁶ (0 prints as 0)."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    return f"{num(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.001 <= v <= 10000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Axis labels in the notebook's number format; call right before plt.show(). Short log axes get 1–2–5 ticks,
    long ones powers of 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def show_row(ax, imgs, title=None):
    """Draw a row of 28 × 28 images (light strokes on a dark background, as in the data)."""
    ax.imshow(torch.cat(list(imgs.reshape(-1, 28, 28)), 1).numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_ylabel(title, rotation=0, ha="right", va="center", fontsize=9)

## 2 · Data: Fashion-MNIST

Fashion-MNIST ([Xiao, Rasul & Vollgraf, 2017](https://arxiv.org/abs/1708.07747), MIT license) has the same format as
MNIST: 60,000 training images and 10,000 test images, 28 × 28 grayscale, ten kinds of clothing, shoes and bags. We
download the four original files from the dataset's repository and check their MD5 (the same checksums torchvision
uses). As in every lesson of the course, the first 55,000 images are for training, the last 5,000 of the training
part are for validation (choosing hyperparameters), and the 10,000 test images are used only to report results. The
same cell loads the script's results and the pretrained weights.

In [ ]:
URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FILES = {  # the official MD5 checksums of Fashion-MNIST
    "train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d",
    "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe",
    "t10k-images-idx3-ubyte.gz": "bef4ecab320f06d8554ea6380940ec79",
    "t10k-labels-idx1-ubyte.gz": "bb300cfdad3c16e7a12a480ee83cd310",
}
DATA = Path("fashion")
DATA.mkdir(exist_ok=True)
for name, md5 in FILES.items():
    p = DATA / name
    if not p.exists():
        urllib.request.urlretrieve(URL + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted: delete it and download it again"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted: delete it and download it again"
    return json.loads(p.read_text())


R = fetch_json("vqvae-results.json", "d4494be0298e4d9852b63f696fef1383")
W = fetch_json("vqvae-weights.json", "cc3977cf197e6f111dc657b546160cb3")

In [ ]:
CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 of a label file) are the header, the rest are bytes 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        y = np.frombuffer(f.read(), np.uint8, offset=8)
    return torch.from_numpy(x.astype(np.float32) / 255), torch.from_numpy(y.astype(np.int64))


X_all, y_all = load("train")
X, V, y_tr = X_all[:55000], X_all[55000:], y_all[:55000]
T, y_te = load("t10k")
print(f"training {tuple(X.shape)} · validation {tuple(V.shape)} · test {tuple(T.shape)}")
print(
    f"pixels exactly 0 (black background): {pct(float((X == 0).float().mean()))} · mean brightness {num(X.mean(), 3)}"
)

fig, axes = plt.subplots(1, 10, figsize=(11, 1.6))
for c, ax in enumerate(axes):
    ax.imshow(T[int((y_te == c).nonzero()[0, 0]), 0], cmap="gray")
    ax.set_title(CLASSES[c], fontsize=8)
    ax.axis("off")
plt.show()

Each panel is the first test image of one class, light strokes on a black background as in the original data; the
class name is on top. Half of all pixels are pure black background: we'll meet this number again when we look at
the most used tokens.

## 3 · Counting bits

The original image has 784 pixels of 8 bits each. Our VQ-VAE shrinks the image 4 times along each side, leaving a
7 × 7 = 49-cell grid; each cell is the index of one of K = 128 codes, that is log₂ 128 = 7 bits. For comparison: a
continuous autoencoder with the same grid, one vector of 16 real numbers of 32 bits per cell as in Lesson 19, takes
*more bits than the original image*. Real numbers are not compression; tokens are.

In [ ]:
K, D = 128, 16
bits_raw = 28 * 28 * 8
bits_float = 7 * 7 * D * 32
bits_vq = 7 * 7 * math.log2(K)
print(f"original image:              {num(bits_raw, 0):>7s} bits")
print(f"7 × 7 × {D} float32 grid:      {num(bits_float, 0):>7s} bits")
print(
    f"49 tokens, each 1 of {K}:      {num(bits_vq, 0):>7s} bits  ({num(bits_raw / bits_vq, 1)} times smaller than the image)"
)
assert bits_vq == 343

## 4 · Vector quantization and straight-through

The encoder is a small CNN (Lesson 11): two stride-2 convolutions take 28 × 28 down to 7 × 7, then a 1 × 1
convolution gives each cell a vector $z_e \in \mathbb{R}^{16}$. **Quantization** replaces $z_e$ with the nearest code
in the codebook $e_1, \dots, e_K$ (the paper's equations 1–2):

$$k = \arg\min_j \lVert z_e - e_j \rVert^2, \qquad z_q = e_k .$$

The distances for every (vector, code) pair are computed at once through the expansion
$\lVert z \rVert^2 - 2\, z \cdot e + \lVert e \rVert^2$: the middle part is a matrix multiplication. The index $k$ is
the token. The decoder mirrors the encoder (transposed convolutions) and ends with a sigmoid, giving pixels in
[0, 1].

**Straight-through.** $\arg\min$ has no derivative, so in the backward pass we *pretend* quantization is the identity
function: the gradient at the decoder's input is copied unchanged to $z_e$. In code that is `ze + (zq − ze).detach()`:
its value is `zq`, while the gradient flows to `ze`.

The cell below defines the model. The `VQ` class also has two things we'll need in section 5: how the codebook is
initialized (`init`) and the function `refresh`, run after every step to replace codes that nobody uses.

In [ ]:
GRID = 7


class VQ(nn.Module):
    """A K × D codebook: nearest code, updated by the loss (eq. 3) or by EMA, dead codes restarted."""

    def __init__(self, K=128, D=16, init="data", update="loss", restart=True, decay=0.99, eps=1e-5):
        super().__init__()
        if init == "randn":
            w = torch.randn(K, D)
        elif init == "uniform":
            w = torch.empty(K, D).uniform_(-1 / K, 1 / K)
        else:
            w = torch.zeros(K, D)  # filled with the encoder's outputs on the first batch (init_from)
        if update == "loss":
            self.book = nn.Parameter(w)
        else:
            self.register_buffer("book", w)
            self.register_buffer("ema_sum", w.clone())
        self.register_buffer("count", torch.ones(K))  # EMA of how often each code is chosen in a batch
        self.K, self.update, self.restart, self.decay, self.eps = K, update, restart, decay, eps
        self.restarts = 0

    def nearest(self, ze):
        """ze: (N, D) → index of the nearest code (squared distance, expanded as ‖z‖² − 2z·e + ‖e‖²)."""
        e = self.book
        return ((ze**2).sum(1, keepdim=True) - 2 * ze @ e.T + (e**2).sum(1)).argmin(1)

    def forward(self, ze):
        idx = self.nearest(ze.detach())
        return self.book[idx], idx

    @torch.no_grad()
    def init_from(self, ze, g):
        self.book.copy_(ze[torch.randperm(len(ze), generator=g)[: self.K]])
        if self.update == "ema":
            self.ema_sum.copy_(self.book)

    @torch.no_grad()
    def refresh(self, ze, idx, g):
        """After every step: count the codes, update the EMA (if used), replace dead codes with random batch outputs."""
        n = torch.bincount(idx, minlength=self.K).float()
        self.count.mul_(self.decay).add_(n, alpha=1 - self.decay)
        if self.update == "ema":
            s = torch.zeros_like(self.ema_sum).index_add_(0, idx, ze)
            self.ema_sum.mul_(self.decay).add_(s, alpha=1 - self.decay)
            tot = self.count.sum()
            N = (self.count + self.eps) / (tot + self.K * self.eps) * tot  # Laplace smoothing, as in Sonnet
            self.book.copy_(self.ema_sum / N[:, None])
        if self.restart:
            dead = self.count < 1
            k = int(dead.sum())
            if k:
                new = ze[torch.randint(len(ze), (k,), generator=g)]
                self.book[dead] = new
                if self.update == "ema":
                    self.ema_sum[dead] = new
                self.count[dead] = 1
                self.restarts += k


class VQVAE(nn.Module):
    def __init__(self, quant="vq", K=128, D=16, ch=32, init="data", update="loss", restart=True):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, ch // 2, 4, 2, 1), nn.ReLU(), nn.Conv2d(ch // 2, ch, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.ReLU(), nn.Conv2d(ch, D, 1),
        )  # fmt: skip
        self.dec = nn.Sequential(
            nn.Conv2d(D, ch, 3, 1, 1), nn.ReLU(), nn.ConvTranspose2d(ch, ch // 2, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(ch // 2, 1, 4, 2, 1),
        )  # fmt: skip
        self.quant, self.d = quant, D
        self.q = VQ(K, D, init, update, restart) if quant == "vq" else None  # "none": a continuous autoencoder

    def encode(self, x):
        """(B, 1, 28, 28) → z_e as (B·49, D), each row the vector of one 4 × 4 cell of the 7 × 7 grid."""
        return self.enc(x).permute(0, 2, 3, 1).reshape(-1, self.d)

    def decode_z(self, z):
        return torch.sigmoid(self.dec(z.reshape(-1, GRID, GRID, self.d).permute(0, 3, 1, 2)))

    def decode(self, idx):
        """Tokens (B, 7, 7) → images: look up the codebook, then decode."""
        return self.decode_z(self.q.book[idx.reshape(-1)])

    def forward(self, x):
        ze = self.encode(x)
        if self.q is None:
            return self.decode_z(ze), ze, ze, None
        zq, idx = self.q(ze)
        z_st = ze + (zq - ze).detach()  # straight-through: the value is zq, the gradient flows to ze
        return self.decode_z(z_st), ze, zq, idx.reshape(-1, GRID, GRID)


m0 = VQVAE(init="randn")
x8 = X[:8]
with torch.no_grad():
    ze = m0.encode(x8)
    d_full = torch.cdist(ze, m0.q.book) ** 2
print(f"8 images → z_e {tuple(ze.shape)} (8 · 49 vectors of 16 dimensions); codebook {tuple(m0.q.book.shape)}")
assert torch.equal(m0.q.nearest(ze), d_full.argmin(1)), "the expansion must pick the same codes as the direct distance"
n_par = sum(p.numel() for n, p in m0.named_parameters() if not n.startswith("q."))
print(f"encoder + decoder: {num(n_par, 0)} parameters; codebook: {num(m0.q.book.numel(), 0)} numbers")

The codebook has only 128 × 16 = 2,048 numbers, far fewer than the network. One question remains: how does the
codebook learn?

### 🤔 Predict first
Train with the reconstruction loss $\lVert x - \hat x \rVert^2$ alone. How large is its gradient with respect to the
codebook?

In [ ]:
torch.manual_seed(0)
m0 = VQVAE(init="randn")
xr, ze, zq, idx = m0(x8)
F.mse_loss(xr, x8).backward()
g_book = 0.0 if m0.q.book.grad is None else float(m0.q.book.grad.abs().max())
print(f"max |∂ℒ_reconstruction/∂codebook| = {sci(g_book)}")
print(f"max |∂ℒ_reconstruction/∂W of the encoder's first layer| = {sci(float(m0.enc[0].weight.grad.abs().max()))}")
assert g_book == 0

**Exactly 0.** Straight-through copies the gradient from the decoder's input to $z_e$, *skipping* the codebook: the
value `zq` sits inside `.detach()`. The encoder and the decoder learn, while the codebook stays put forever. That's
why the paper adds two terms, where $\mathrm{sg}[\cdot]$ is the *stop-gradient* (the value is unchanged, the
derivative is 0):

$$\mathcal{L} = \lVert x - \hat x \rVert^2 + \lVert \mathrm{sg}[z_e] - e_k \rVert^2
+ \beta \lVert z_e - \mathrm{sg}[e_k] \rVert^2 .$$

The middle term (the *codebook loss*) pulls the chosen code toward $z_e$, like a k-means step. The last term (the
*commitment loss*) pulls $z_e$ toward the chosen code, so that the encoder "commits" to a code instead of drifting
away; the paper uses β = 0.25. The two terms look alike but are **not symmetric**: each one trains only one side. The
cell below asks autograd where each term sends its gradient (it prints the largest gradient magnitude; 0 means no
gradient).

In [ ]:
def loss_terms(model, x, beta):
    """The paper's loss (eq. 3, as means): reconstruction + codebook (‖sg[ze] − e‖²) + β · commitment (‖ze − sg[e]‖²)."""
    xr, ze, zq, idx = model(x)
    rec = F.mse_loss(xr, x)
    if model.q is None:
        return rec, rec, ze, idx
    code = F.mse_loss(zq, ze.detach()) if model.q.update == "loss" else torch.zeros(())
    commit = F.mse_loss(ze, zq.detach())
    return rec + code + beta * commit, rec, ze, idx


def grad_size(t):
    return 0.0 if t.grad is None else float(t.grad.abs().max())


xr, ze, zq, idx = m0(x8)
terms = {
    "reconstruction": F.mse_loss(xr, x8),
    "codebook": F.mse_loss(zq, ze.detach()),
    "commitment": F.mse_loss(ze, zq.detach()),
}
print(f"{'term':15s}{'encoder':>12s}{'codebook':>12s}{'decoder':>12s}")
table = {}
for name, term in terms.items():
    m0.zero_grad()
    term.backward(retain_graph=True)
    table[name] = [grad_size(m0.enc[0].weight), grad_size(m0.q.book), grad_size(m0.dec[0].weight)]
    print(f"{name:15s}" + "".join(f"{sci(v):>12s}" for v in table[name]))
assert table["reconstruction"][1] == 0 and table["codebook"][0] == table["codebook"][2] == 0
assert table["commitment"][1] == table["commitment"][2] == 0 and table["commitment"][0] > 0
m0.zero_grad()
terms["codebook"].backward()
moved = int((m0.q.book.grad.abs().sum(1) > 0).sum())
print(f"the codebook loss touches only {moved} of the {K} codes: those chosen by at least one vector")
assert moved == len(torch.unique(idx))

The table is exactly what the equation says: reconstruction trains the encoder (through straight-through) and the
decoder; the codebook loss trains only the codebook; commitment trains only the encoder. The last line is the seed of
the trouble in the next section: **only the codes that get chosen receive a gradient**, and with this N(0, 1)
codebook, all 392 vectors of the 8 images crowd into a handful of codes (last line). A code that is never the nearest
code of any vector never learns, and is never used.

The paper also has a variant without the codebook loss (Appendix A.1, not used in the paper's experiments): update
each code as an exponential moving average (EMA, factor γ = 0.99) of the vectors that chose it, that is, online
k-means. The `VQ` class above has both (`update="loss"` or `"ema"`); section 9 compares them.

## 5 · A codebook dead from initialization

Many implementations initialize the codebook from $\mathcal{N}(0, 1)$ (for example `zalandoresearch/pytorch-vq-vae`
for its EMA version, and `rosinality/vq-vae-2-pytorch`), others from the uniform distribution $U(-1/K, 1/K)$
(taming-transformers, Zalando's non-EMA version).

### 🤔 Predict first
A codebook of 128 codes initialized from $\mathcal{N}(0, 1)$, network untrained. Pass 64 images through the encoder:
64 · 49 = 3,136 vectors. How many different codes do they choose?

In [ ]:
torch.manual_seed(0)
m0 = VQVAE(init="randn")
g = torch.Generator().manual_seed(0)
xb = X[torch.randint(len(X), (64,), generator=g)]
with torch.no_grad():
    ze = m0.encode(xb)
    books = {
        "N(0, 1)": m0.q.book.clone(),
        "U(−1/K, 1/K)": torch.empty(K, D).uniform_(-1 / K, 1 / K),
        "from data": ze[torch.randperm(len(ze), generator=g)[:K]],
    }
    print(f"{num(len(ze), 0)} z_e vectors, mean length at initialization {num(ze.norm(dim=1).mean(), 3)}")
    used0 = {}
    for name, book in books.items():
        m0.q.book.data.copy_(book)
        used0[name] = len(torch.unique(m0.q.nearest(ze)))
        print(
            f"codebook {name:13s} code length ~{num(book.norm(dim=1).mean(), 3):>6s} → {used0[name]:3d} / {K} codes chosen"
        )
assert used0["N(0, 1)"] <= 5 and used0["from data"] > 100

Just a handful of codes. At initialization the $z_e$ vectors are short (length about 0.4) and huddle near the
origin, while the 16-dimensional $\mathcal{N}(0, 1)$ codes have length about $\sqrt{16} = 4$, scattered over a
faraway sphere. For almost every $z_e$, the nearest code is the same one or two codes that happen to lie closest to
the origin. The other codes are never chosen, so they get no gradient (section 4), so they are never chosen. The
uniform $U(\pm 1/K)$ codebook is the opposite: every code sits right at the origin, tens of times shorter than $z_e$,
and again only the few codes that happen to point toward the $z_e$ get chosen.

Two common fixes, both already written into the `VQ` class:

- **Initialize from data:** use K random $z_e$ vectors of the first batch as the codebook (Karpathy and SoundStream
  run k-means on the first batch; we only sample at random, like the starting step of k-means).
- **Restart dead codes** (Jukebox, Dhariwal et al., 2020): `refresh` keeps a moving average of how often each code is
  chosen per batch; a code that falls below 1 is reset to a random $z_e$ of the current batch.

At initialization, the "from data" codebook already uses all 128 codes. But can training fix a bad start on its own?
The cell below loads the script's results: 3,000 steps, 3 seeds per option, and plots the number of codes used on
the validation set against the step.

In [ ]:
INIT_RUNS = {
    "randn": "N(0, 1), no restarts",
    "uniform": "U(±1/K), no restarts",
    "data": "from data, no restarts",
    "randn_restart": "N(0, 1) + restarts",
    "ref": "from data + restarts (reference)",
}
STYLE = {
    "randn": (ORANGE, "o", "-"),
    "uniform": (ORANGE, "s", "--"),
    "data": (GREY, "^", "-"),
    "randn_restart": (BLUE, "D", "--"),
    "ref": (GREEN, "o", "-"),
}


def seeds(name, key="mse", where="test"):
    return np.array([run[where][key] for run in R["runs"][name]], dtype=float)


def span(a, digits=4, f=num):
    a = np.asarray(a, dtype=float)
    return f(a.min(), digits) if np.isclose(a.min(), a.max()) else f"{f(a.min(), digits)}–{f(a.max(), digits)}"


STEPS = R["steps"]
steps_axis = [int(s) for s in R["runs"]["ref"][0]["checks"]]
fig, ax = plt.subplots(figsize=(9.5, 3.6))
for name, label in INIT_RUNS.items():
    c, mk, ls = STYLE[name]
    u = np.array([[run["checks"][str(s)]["used"] for s in steps_axis] for run in R["runs"][name]])
    ax.fill_between(steps_axis, u.min(0), u.max(0), color=c, alpha=0.15)
    ax.plot(steps_axis, np.median(u, 0), marker=mk, ls=ls, color=c, label=label)
ax.set(xscale="log", xlabel="training step", ylabel="codes used (out of 128)", ylim=(0, 135))
ax.set_title("codes used on the validation set (median and range over 3 seeds)", fontsize=10)
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
fmt_axes(fig)
plt.tight_layout()
plt.show()
AT = (100, 400, 1000, 2000, STEPS)
print(f"{'codes used (validation), step':38s}" + "".join(f"{num(s, 0):>12s}" for s in AT) + f"{'test MSE':>17s}")
for name, label in INIT_RUNS.items():
    used = [span([run["checks"][str(s)]["used"] for run in R["runs"][name]], 0) for s in AT]
    print(f"{label:38s}" + "".join(f"{u:>12s}" for u in used) + f"{span(seeds(name)):>17s}")

Three things to read from the plot and the table:

- **N(0, 1) doesn't fix itself.** After 3,000 steps only a few codes are still in use (solid orange line), and the
  MSE is twice the reference model's. The codebook is still there, but the encoder has only a few "words" to speak
  with.
- **U(±1/K) fixes itself, but slowly.** As training makes the $z_e$ longer, they pull the codes near the origin out
  one by one, and the number of codes used rises gradually (dashed orange line). At step 400 it doesn't use even a
  third of the codebook: a tutorial that trains briefly would conclude that this initialization "causes collapse",
  when really it is only slow.
- **Restarting dead codes cures even the worst start.** N(0, 1) plus restarts (dashed blue line) uses all 128 codes
  from step 1,000 on, and its MSE is within the seed spread of the reference model. Initializing from data without
  restarts does fairly well, but still leaves a few dead codes and a higher MSE.

## 6 · Training a VQ-VAE

The reference model: K = 128 codes, D = 16, β = 0.25, the paper's loss, initialization from data and dead-code
restarts, Adam with batch 64. The learning rate η was chosen by the script on the validation set with seed 0, then
shared by every configuration. We run the first 400 steps of that exact seed-0 run live; every step draws its batch,
the codebook initialization and the restarts from the same seeded random generator, so the numbers must match the
script.

In [ ]:
REF = R["ref"]
LIVE = R["live"]
etas = ", ".join(f"{tick(e)} → {num(v, 5)}" for e, v in zip(R["eta"]["grid"], R["eta"]["val"], strict=True))
print(f"validation MSE after {num(R['steps'], 0)} steps for each η: {etas}; chosen η = {tick(REF['lr'])}")


def train(cfg, seed, steps, log=None):
    torch.manual_seed(seed)  # initial weights
    g = torch.Generator().manual_seed(seed)  # batches, codebook initialization, dead-code restarts
    m = VQVAE(cfg["quant"], cfg["K"], cfg["D"], 32, cfg["init"], cfg["update"], cfg["restart"])
    opt = torch.optim.Adam(m.parameters(), cfg["lr"])
    bs = cfg["bs"]
    if m.q is not None and cfg["init"] == "data":
        with torch.no_grad():
            m.q.init_from(m.encode(X[torch.randint(len(X), (bs,), generator=g)]), g)
    for step in range(1, steps + 1):
        x = X[torch.randint(len(X), (bs,), generator=g)]
        loss, rec, ze, idx = loss_terms(m, x, cfg["beta"])
        opt.zero_grad()
        loss.backward()
        opt.step()
        if m.q is not None:
            m.q.refresh(ze.detach(), idx.reshape(-1), g)
        if log is not None and step % 50 == 0:
            log.append(rec.item())
    return m.eval()


@torch.no_grad()
def metrics(m, X, chunk=2000):
    """Mean MSE per pixel, PSNR, number of codes used, perplexity exp(H) of the code frequencies."""
    se, codes = [], []
    for i in range(0, len(X), chunk):
        x = X[i : i + chunk]
        xr, _, _, idx = m(x)
        se.append(((xr - x) ** 2).mean((1, 2, 3)))
        if idx is not None:
            codes.append(idx.reshape(-1))
    mse = float(torch.cat(se).mean())
    out = {"mse": mse, "psnr": 10 * math.log10(1 / mse)}
    if codes:
        p = torch.bincount(torch.cat(codes), minlength=m.q.K).double() / sum(len(c) for c in codes)
        nz = p[p > 0]
        out.update(used=int((p > 0).sum()), ppl=float(torch.exp(-(nz * nz.log()).sum())))
    return out


live_curve = []
live = train(REF, 0, LIVE, live_curve)
got = metrics(live, V)
want = R["runs"]["ref"][0]["checks"][str(LIVE)]
print(f"after {LIVE} steps (validation): MSE {num(got['mse'], 5)} (script {num(want['mse'], 5)}) · "
      f"{got['used']} codes used (script {int(want['used'])}) · perplexity {num(got['ppl'], 1)} (script {num(want['ppl'], 1)}) · "
      f"{live.q.restarts} codes restarted")  # fmt: skip
assert abs(got["mse"] / want["mse"] - 1) < 0.05 and abs(got["used"] - want["used"]) <= 4, "differs from the script"
assert abs(got["ppl"] / want["ppl"] - 1) < 0.1, "differs from the script"

**Perplexity** is $\exp(H)$, where $H$ is the entropy of the code frequencies: the "effective" number of codes. It
equals the number of codes used if every used code is used equally often, and is smaller when a few codes dominate.

On another machine (Colab runs on x86 CPUs), floating-point operations can round differently in the last digit, and
a vector right on the boundary between two codes can pick the other one; that's why the cell above compares with the
script within a small tolerance instead of demanding exact equality.

The plot below draws the reconstruction loss on the training batches of the run you just did, on top of the
script's 3,000-step run. Then we load that very 3,000-step model (stored in float16) and measure it on the test set.

In [ ]:
ref_curve = R["runs"]["ref"][0]["curve"]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(np.arange(1, len(ref_curve) + 1) * 50, ref_curve, color=GREY, label=f"script, {num(STEPS, 0)} steps")
ax.plot(np.arange(1, len(live_curve) + 1) * 50, live_curve, "o", ms=4, color=BLUE, label=f"live run, {LIVE} steps")
ax.set(yscale="log", xlabel="step", ylabel="MSE on the training batch")
ax.legend()
fmt_axes(fig)
plt.tight_layout()
plt.show()


def unpack(p):
    """Unpack float16 (base64) weights into a state dict."""
    return {
        k: torch.tensor(np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float32).reshape(v["shape"]))
        for k, v in p.items()
    }


vqm = VQVAE()
vqm.load_state_dict(unpack(W["vq"]))
vqm.eval()
aem = VQVAE("none")
aem.load_state_dict(unpack(W["ae"]))
aem.eval()
test = metrics(vqm, T)
want = R["runs"]["ref"][0]["test"]
print(f"the {num(STEPS, 0)}-step model on the test set: MSE {num(test['mse'], 5)} · PSNR {num(test['psnr'], 2)} dB · "
      f"{test['used']} / {K} codes · perplexity {num(test['ppl'], 1)}")  # fmt: skip
assert abs(test["mse"] / want["mse"] - 1) < 0.01 and abs(test["used"] - want["used"]) <= 1
assert abs(test["ppl"] / want["ppl"] - 1) < 0.02

The figure below puts side by side: the original test images, the model you just trained for 400 steps, the
3,000-step model, and a continuous autoencoder with the same 7 × 7 × 16 grid, the same network and exactly the same
training, but no quantization (section 9 compares the two more closely).

In [ ]:
pick = torch.tensor([int((y_te == c).nonzero()[1, 0]) for c in range(10)])
with torch.no_grad():
    rows = [T[pick], live(T[pick])[0], vqm(T[pick])[0], aem(T[pick])[0]]
names = ["original", f"VQ-VAE, {LIVE} steps", f"VQ-VAE, {num(STEPS, 0)} steps", "continuous autoencoder"]
fig, axes = plt.subplots(4, 1, figsize=(10, 4.6))
for ax, row, name in zip(axes, rows, names, strict=True):
    show_row(ax, row, name)
plt.tight_layout()
plt.show()

After 400 steps, the model already keeps the shape of each item (trouser legs, sleeves, the bag's handle) but is
still blurry. After 3,000 steps the outlines and the large details, such as the pullover's hem or the sandal's sole,
are sharper, but the shirt's check pattern is still lost: 7 bits for a 4 × 4 cell can't hold it. The continuous
autoencoder with the same grid keeps even that pattern.

## 7 · What does a token carry?

The cell below encodes the 10,000 test images into tokens (only the encoder and the nearest-code step are needed),
then prints the token grid of a top.

In [ ]:
@torch.no_grad()
def codes(m, X, chunk=5000):
    return torch.cat([m.q.nearest(m.encode(X[i : i + chunk])) for i in range(0, len(X), chunk)]).view(-1, GRID, GRID)


C_te = codes(vqm, T)
ex = int(pick[0])
print(f"test image #{ex} ({CLASSES[int(y_te[ex])]}): 49 tokens")
for row in C_te[ex].tolist():
    print("   " + " ".join(f"{c:3d}" for c in row))
flat_c = C_te.reshape(-1)
cnt = torch.bincount(flat_c, minlength=K)
cells = T[:, 0].unfold(1, 4, 4).unfold(2, 4, 4).mean((-1, -2))  # mean brightness of each 4 × 4 cell: (10000, 7, 7)
cell_mean = torch.zeros(K).index_add_(0, flat_c, cells.reshape(-1)) / cnt.clamp_min(1)
is_bg = (cell_mean < 0.02) & (cnt > 0)  # "background" codes: the cell underneath is almost always black
fg = ~is_bg & (cnt > 0)
bg = int(torch.where(is_bg, cnt, -1).argmax())  # the most common background code
print(f"{int((cnt > 0).sum())} codes occur; {int(is_bg.sum())} of them are 'background' codes (the cell underneath has a "
      f"mean brightness below 0.02), together {pct(float(cnt[is_bg].sum() / cnt.sum()))} of all cells")  # fmt: skip
print(f"most used code: {int(cnt.argmax())}, {pct(float(cnt.max() / cnt.sum()))} of all cells")
here = sorted({c for c in C_te[ex].flatten().tolist() if is_bg[c]})
print(f"background codes in the grid of the top above: {here}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={"width_ratios": [1, 2.2]})
axes[0].imshow(T[ex, 0], cmap="gray", extent=(0, 7, 7, 0))
for i in range(GRID):
    for j in range(GRID):
        c = int(C_te[ex, i, j])
        axes[0].text(j + 0.5, i + 0.5, str(c), ha="center", va="center", fontsize=6.5,
                     color="black" if T[ex, 0, 4 * i : 4 * i + 4, 4 * j : 4 * j + 4].mean() > 0.5 else "white")  # fmt: skip
axes[0].set(xticks=[], yticks=[], title="the token of each 4 × 4 cell")
order = torch.argsort(cnt, descending=True)
frac = 100 * cnt[order].numpy() / cnt.sum().item()
bgo = is_bg[order].numpy()
axes[1].bar(np.arange(K)[~bgo], frac[~bgo], color=BLUE, width=1.0, label="codes with strokes")
axes[1].bar(
    np.arange(K)[bgo], frac[bgo], color=GREY, width=1.0, hatch="///", ec="white", lw=0, label="background codes"
)
axes[1].legend(fontsize=8)
axes[1].set(
    yscale="log",
    xlabel="code, ranked by frequency",
    ylabel="% of cells",
    title="frequency of the 128 codes on the test set",
)
fmt_axes(fig)
plt.tight_layout()
plt.show()

The token grid on the left shows two things. First, the item's *edges* are encoded consistently: the top's left edge
is the same code for six rows in a row. Second, the black background on either side doesn't share one code but uses
more than ten different codes (the last printed line in the cell above). The whole model has dozens of such
"background" codes, together more than a third of all cells. The culprit is the very fix of section 5: a dead code is
reset to a random $z_e$ of the batch, and half of all pixels are background, so new codes tend to land in the
background region and split up cells that need only one code. "All 128 codes in use" and a perplexity of about 100
therefore overstate how many truly different "words" there are; Huh et al. (2023) also warn that a high perplexity can
just be redundancy. On the right, the background codes (grey, hatched) sit both at the top of the ranking and
scattered further down, and no code dominates.

So does a code stand for a kind of garment, the way a word stands for a concept? The cell below counts, for each code
that isn't background, the share of its occurrences that fall in its most frequent class, then draws six codes spread
from most to least frequent.

In [ ]:
M = torch.bincount(flat_c * 10 + y_te.repeat_interleave(GRID * GRID), minlength=K * 10).view(K, 10).float()
purity_fg = float(M[fg].max(1).values.sum() / M[fg].sum())
best = int(torch.argmax(torch.where(fg & (cnt > 500), M.max(1).values / M.sum(1).clamp_min(1), torch.tensor(0.0))))
print(f"for the other {int(fg.sum())} codes: on average {pct(purity_fg)} of a code's occurrences fall in its most "
      f"frequent class (chance: 10%)")  # fmt: skip
share = M[best] / M[best].sum()
print(
    f"the 'purest' code among those seen over 500 times: code {best}, {pct(float(share.max()))} {CLASSES[int(share.argmax())]}"
)

ranked = [int(k) for k in order if fg[k]]
top = [ranked[r] for r in range(0, len(ranked), len(ranked) // 6)][:6]  # 6 codes with strokes, spread by frequency
pad = F.pad(T[:, 0], (6, 6, 6, 6))
g = torch.Generator().manual_seed(0)
fig, axes = plt.subplots(len(top), 1, figsize=(8.5, 7))
for ax, k in zip(axes, top, strict=True):
    where = (C_te == k).nonzero()
    crops = torch.stack([pad[n, 4 * i : 4 * i + 16, 4 * j : 4 * j + 16] for n, i, j in where.tolist()])
    show = [crops.mean(0)] + list(crops[torch.randperm(len(crops), generator=g)[:8]])
    strip = torch.cat([F.pad(c, (1, 1, 1, 1), value=0.5) for c in show], 1)
    ax.imshow(strip.numpy(), cmap="gray", vmin=0, vmax=1)
    for s in range(len(show)):
        ax.add_patch(plt.Rectangle((18 * s + 6.5, 6.5), 4, 4, fill=False, ec=ORANGE, lw=1.2))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylabel(f"code {k}\n({num(cnt[k], 0)} cells)", rotation=0, ha="right", va="center", fontsize=9)
axes[0].set_title("first panel: mean over every occurrence · next 8: random occurrences", fontsize=9)
plt.tight_layout()
plt.show()

Only 16.5% (chance: 10%): a code doesn't say "this is a pullover". The figure above shows what a code does say. Each
row is one code: the first panel is the mean of the 16 × 16-pixel region around every occurrence of the code, the
next eight are random occurrences, and the orange box is the token's 4 × 4 cell. Each code is a **local pattern**: an
even patch of light fabric, a patch of dark fabric, a vertical edge light on one side and dark on the other, a
horizontal edge. The same edge can belong to a shirt, a trouser or a bag. Even the "purest" of the common codes
falls in a single class only about half of the time. The meaning "a whole garment" isn't in any single token but in
how the 49 tokens sit next to each other, and that's what the prior of section 10 has to learn.

## 8 · One token for the whole image

Václav Volhejn's article [*Neural audio codecs: how to get audio into LLMs*](https://kyutai.org/codec-explainer/)
(Kyutai, 2025) also builds a VQ-VAE on Fashion-MNIST before moving on to audio, but very differently from sections
4–7: only the first three classes (T-shirt/top, trouser, pullover), a fully connected network instead of a CNN, and
the whole image becomes **one** vector of 2 numbers, that is **one token for the whole image**. The article takes
four steps, and we rebuild all four with the very `VQ` class of section 4:

1. Train a plain autoencoder: each image becomes 2 numbers.
2. Put a clustering *on top* of that 2-dimensional space, like k-means: each code moves gradually toward the mean of
   the points that chose it, and a code nobody has used for a while is moved to a random point of the batch. That is
   exactly the EMA version plus dead-code restarts of the `VQ` class, run on fixed vectors, with no gradient.
3. Quantize *during training*, with straight-through and the commitment loss, so that the autoencoder itself learns
   to be easy to quantize.
4. **Residual vector quantization** (RVQ): quantize the residual $z - z_q$ with a second codebook, so each image
   becomes a pair of tokens. This is how the audio tokenizers of Topic 05 stack several codebooks.

The network has 4 layers on each side (784 → 128 → 32 → 8 → 2, then back), and each level has K = 32 codes, that is
5 bits. The cell below defines the model and its training (copied verbatim from the script, like the classes of
section 4), then trains the plain autoencoder live for the first 1,000 steps of seed 0 on the training images of the
three classes.

In [ ]:
class MLPAE(nn.Module):
    """Fully connected autoencoder: the whole 28 × 28 image becomes D numbers (4 layers per side); q = VQ if quantized
    during training."""

    def __init__(self, hidden=(128, 32, 8), D=2, K=32, vq=False):
        super().__init__()
        sizes = [28 * 28, *hidden, D]
        enc, dec = [], []
        for a, b in itertools.pairwise(sizes):
            enc += [nn.Linear(a, b), nn.ReLU()]
        for a, b in itertools.pairwise(sizes[::-1]):
            dec += [nn.Linear(a, b), nn.ReLU()]
        self.enc, self.dec = nn.Sequential(*enc[:-1]), nn.Sequential(*dec[:-1], nn.Sigmoid())
        self.q = VQ(K, D, "data", "ema", True) if vq else None

    def encode(self, x):
        return self.enc(x.reshape(len(x), -1))

    def decode_z(self, z):
        return self.dec(z).reshape(-1, 1, 28, 28)


def train_toy(cfg, X, seed, vq=False):
    """Plain autoencoder (vq=False) or quantized during training: straight-through, commitment β, EMA codes + restarts."""
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    m = MLPAE(cfg["hidden"], cfg["D"], cfg["K"], vq)
    opt = torch.optim.Adam(m.parameters(), cfg["lr"])
    if vq:
        with torch.no_grad():
            m.q.init_from(m.encode(X[torch.randint(len(X), (cfg["bs"],), generator=g)]), g)
    for _ in range(cfg["steps"]):
        x = X[torch.randint(len(X), (cfg["bs"],), generator=g)]
        ze = m.encode(x)
        if vq:
            zq, idx = m.q(ze)
            loss = F.mse_loss(m.decode_z(ze + (zq - ze).detach()), x) + cfg["beta"] * F.mse_loss(ze, zq.detach())
        else:
            loss = F.mse_loss(m.decode_z(ze), x)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if vq:
            m.q.refresh(ze.detach(), idx, g)
    return m.eval()


TOY = R["toy"]["cfg"]
TOY_LR = {k: R["toy"]["eta"][k]["chosen"] for k in ("ae", "vq")}
X3, T3, y3 = X[y_tr < TOY["classes"]], T[y_te < TOY["classes"]], y_te[y_te < TOY["classes"]]
t_ae = train_toy({**TOY, "lr": TOY_LR["ae"], "steps": R["toy"]["live"]}, X3, 0)
print(f"{', '.join(CLASSES[:3])}: {num(len(X3), 0)} training images, {num(len(T3), 0)} test images · network "
      f"{num(sum(p.numel() for p in t_ae.parameters()), 0)} parameters · η = {tick(TOY_LR['ae'])} (chosen on "
      f"validation; the version quantized during training: η = {tick(TOY_LR['vq'])})")  # fmt: skip
assert sum(p.numel() for p in t_ae.parameters()) == R["toy"]["runs"]["ae"][0]["params"]

Now steps 2 and 4: `fit_rvq` puts codebooks on top of the training images' 2-number vectors. Each level is 1,000
steps of the `VQ` class on 256 random vectors (nearest code, EMA, dead-code restarts); each next level clusters the
residual the earlier levels left behind. `rvq_encode` encodes a vector in exactly that order: find the nearest code,
subtract it, search again on what's left. The cell below encodes the 3,000 test images with 1 to 4 levels and
compares with the script.

In [ ]:
@torch.no_grad()
def fit_rvq(Z, K, levels, g, steps=1000, bs=256):
    """Clustering *after* training, on fixed vectors Z (n, D): each level is online k-means (the VQ class: nearest
    code, EMA, dead-code restarts, no gradient) on the residual the earlier levels left behind."""
    books, r = [], Z.clone()
    for _ in range(levels):
        q = VQ(K, Z.shape[1], "data", "ema", True)
        q.init_from(r[torch.randint(len(r), (bs,), generator=g)], g)
        for _ in range(steps):
            z = r[torch.randint(len(r), (bs,), generator=g)]
            q.refresh(z, q.nearest(z), g)
        r = r - q.book[q.nearest(r)]
        books.append(q)
    return books


@torch.no_grad()
def rvq_encode(books, z):
    """z (n, D) → codes (n, levels) and the sum of the chosen codes (n, D), taken level by level."""
    out, zq = [], torch.zeros_like(z)
    for q in books:
        i = q.nearest(z - zq)
        out.append(i)
        zq = zq + q.book[i]
    return torch.stack(out, 1), zq


def toy_purity(idx, y, K, classes):
    """Share of a code's occurrences that fall in its most frequent class (averaged over occurrences)."""
    M = torch.bincount(idx * classes + y, minlength=K * classes).view(K, classes)
    return float(M.max(1).values.sum() / M.sum())


with torch.no_grad():
    Z3, Zt3 = t_ae.encode(X3), t_ae.encode(T3)
books = fit_rvq(Z3, TOY["K"], TOY["levels"], torch.Generator().manual_seed(1000), TOY["fitSteps"], TOY["fitBs"])
tc, _ = rvq_encode(books, Zt3)  # (3,000, 4): 4 tokens per test image, each 1 of 32
with torch.no_grad():
    t_mse = [float(((t_ae.decode_z(rvq_encode(books[:n], Zt3)[1]) - T3) ** 2).mean()) for n in range(1, 5)]
    t_ceil = float(((t_ae.decode_z(Zt3) - T3) ** 2).mean())
t_used = [len(tc[:, :n].unique(dim=0)) for n in range(1, 5)]
want = R["toy"]["runs"]["ae"][0]["checks"][str(R["toy"]["live"])]["test"]
print(f"no quantization (2 real numbers): MSE {num(t_ceil, 5)} (script {num(want['mse'], 5)})")
for n in range(4):
    print(f"{n + 1} level{'s' if n else ' '} = {5 * (n + 1):2d} bits per image: MSE {num(t_mse[n], 5)} (script "
          f"{num(want['rvq'][n], 5)}) · {num(t_used[n], 0):>5s} distinct reconstructions for {num(len(T3), 0)} test images")  # fmt: skip
assert abs(t_ceil / want["mse"] - 1) < 0.02 and t_used[0] == want["used"][0] == TOY["K"], "differs from the script"
assert all(abs(a / b - 1) < 0.03 for a, b in zip(t_mse, want["rvq"], strict=True)), "differs from the script"
pur1 = toy_purity(tc[:, 0], y3, TOY["K"], 3)
pur49 = toy_purity(C_te[y_te < 3].reshape(-1), y3.repeat_interleave(GRID * GRID), K, 3)
print(f"a first-level token falls in its most frequent class {pct(pur1)} of the time (chance: {pct(1 / 3)}); "
      f"the 49 grid tokens of section 7, on the same three classes: {pct(pur49)}")  # fmt: skip
assert abs(pur1 - want["purity"]) < 0.02, "differs from the script"
print(f"the 7 × 7 grid VQ-VAE (343 bits) on the same {num(len(T3), 0)} images: MSE {num(metrics(vqm, T3)['mse'], 5)}")

One token for the whole image means the entire test set can only be rebuilt as **32 different images**: two images
with the same token come out as exactly the same image. With a second level, the number of distinct reconstructions
grows to hundreds and the MSE drops clearly; by the fourth level, the MSE nearly touches that of the 2 unquantized
real numbers. The article says exactly this: the best possible quality is the reconstruction from the unquantized
code, and if the autoencoder is poor, no amount of clever quantization will save it. Here that ceiling is low: 2
numbers for a whole image are still far blurrier than the grid's 49 tokens (last line).

In exchange, each token now nearly names the garment: more than 90% of the time, a code falls in one class, while a
grid token, on the same three classes, does so only about half the time. The figure below shows why. Left: the
2-number vector of every test image, each class with its own color and marker (the autoencoder never sees the
labels, yet arranges the three classes into three regions), together with the 32 first-level codes (black plus
signs). Right: the 32 images the decoder draws from those 32 codes, sorted by the code's most frequent class, then by
$z_1$.

In [ ]:
lv1 = books[0].book  # the 32 first-level codes, 2 numbers each
maj = torch.bincount(tc[:, 0] * 3 + y3, minlength=TOY["K"] * 3).view(TOY["K"], 3).argmax(1)
order1 = sorted(range(TOY["K"]), key=lambda k: (int(maj[k]), float(lv1[k, 0])))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1.1, 1]})
for c, (col, mk) in enumerate(((BLUE, "o"), (ORANGE, "s"), (GREEN, "^"))):
    z = Zt3[y3 == c]
    axes[0].scatter(z[:, 0], z[:, 1], s=7, color=col, marker=mk, alpha=0.45, lw=0, label=CLASSES[c])
axes[0].scatter(lv1[:, 0], lv1[:, 1], s=60, color="black", marker="+", label="the 32 first-level codes")
axes[0].set(xlabel="z₁", ylabel="z₂", title=f"{num(len(T3), 0)} test images, 2 numbers each")
axes[0].legend(fontsize=8, markerscale=1.5)
with torch.no_grad():
    proto = t_ae.decode_z(lv1[order1])[:, 0]
axes[1].imshow(torch.cat([torch.cat(list(proto[r * 8 : r * 8 + 8]), 1) for r in range(4)]).numpy(), cmap="gray")
axes[1].set(xticks=[], yticks=[], title="the only 32 images one token decodes to")
fmt_axes(fig)
plt.tight_layout()
plt.show()
print("codes by their most frequent class:", " · ".join(f"{CLASSES[c]} {int((maj == c).sum())}" for c in range(3)))

The three classes sit in three regions of the plane, and the codes share out each region, so each code is a
"template image" of one style of top or trouser: a light or dark, wide or narrow T-shirt; wide or narrow trousers; a
light or dark pullover. That is a token in a sense close to a *word*. A grid token of section 7, by contrast, is a
4 × 4 patch of pixels: less "meaning", but 49 of them together keep far more detail.

The figure below redoes the article's comparison figure. The cell below finds two images with the same first token
but different second tokens, one image with the same second token as the first image, and one image of another
class; under each column is its pair of tokens (level 1, level 2).

In [ ]:
i1, i2 = tc[:, 0], tc[:, 1]
for a in range(len(T3)):
    same1 = ((i1 == i1[a]) & (i2 != i2[a])).nonzero().flatten()
    same2 = ((i2 == i2[a]) & (i1 != i1[a])).nonzero().flatten()
    if len(same1) and len(same2):
        break
cols = [a, int(same1[0]), int(same2[0]), int(((y3 != y3[a]) & (i1 != i1[a])).nonzero()[0, 0])]
with torch.no_grad():
    z_one, z_two = books[0].book[i1[cols]], books[0].book[i1[cols]] + books[1].book[i2[cols]]
    rows = [T3[cols], t_ae.decode_z(Zt3[cols]), t_ae.decode_z(z_one), t_ae.decode_z(z_two)]
fig, axes = plt.subplots(4, 1, figsize=(4.6, 5.4))
for ax, row, name in zip(
    axes, rows, ["original", "2 real numbers", "1 level (5 bits)", "2 levels (10 bits)"], strict=True
):
    show_row(ax, row, name)
axes[-1].set_xticks([14 + 28 * k for k in range(4)], [f"({int(i1[n])}, {int(i2[n])})" for n in cols], fontsize=8)
plt.tight_layout()
plt.show()
print(f"columns 1 and 2 share the first token: their 1-level reconstructions differ by at most "
      f"{num(float((rows[2][0] - rows[2][1]).abs().max()), 3)}")  # fmt: skip
step2 = rows[3] - rows[2]  # how the second level changes the pixels
print(f"columns 1 and 3 share the second token (the same vector added to z), yet change the pixels differently: mean "
      f"difference {num(float((step2[0] - step2[2]).abs().mean()), 3)} against a mean change of {num(float(step2[[0, 2]].abs().mean()), 3)}")  # fmt: skip

The first two columns share the first token, so in the "1 level" row they are exactly the same image; the second
token tells them apart, if only slightly. Columns 1 and 3 share the second token, that is the same vector added to
$z$, yet the decoder turns that same vector into different pixel changes: the second level corrects the *code*, not
the pixels directly, just as the article stresses.

### 🤔 Predict first
So far we have only *put* a codebook on top of an autoencoder that had finished training; the autoencoder never knew
it would be quantized. The article's step 3 quantizes during training (straight-through and commitment, as in
section 4), so that the model itself learns to be easy to quantize. With the same 32 codes, which one reconstructs
better? Does the answer change with training time?

In [ ]:
TR = R["toy"]["runs"]
t_steps = [int(s) for s in TR["ae"][0]["checks"]]
LINES = [  # (kind, key, level, label, color, marker, line style)
    ("ae", "mse", None, "plain, no quantization (ceiling)", GREY, "o", "--"),
    ("ae", "rvq", 0, "plain + clustering on top, 1 level", ORANGE, "s", "-"),
    ("vq", "mseQ", None, "quantized during training, 1 level", GREEN, "D", "-"),
    ("ae", "rvq", 1, "plain + clustering, 2 levels", ORANGE, "s", ":"),
    ("vq", "mseQ2", None, "during training, + 1 clustered level", GREEN, "D", ":"),
    ("vq", "mse", None, "during training, no quantization (ceiling)", GREEN, "^", "--"),
]


def toy_seeds(kind, key, j, step):
    v = [run["checks"][str(step)]["test"][key] for run in TR[kind]]
    return np.array(v if j is None else [x[j] for x in v], dtype=float)


fig, ax = plt.subplots(figsize=(9.5, 3.8))
print(f"{'test MSE (3 seeds), step':44s}" + "".join(f"{num(s, 0):>17s}" for s in t_steps))
for kind, key, j, label, c, mk, ls in LINES:
    a = np.array([toy_seeds(kind, key, j, s) for s in t_steps])  # (step, seed)
    ax.fill_between(t_steps, a.min(1), a.max(1), color=c, alpha=0.12)
    ax.plot(t_steps, np.median(a, 1), color=c, marker=mk, ls=ls, label=label)
    print(f"{label:44s}" + "".join(f"{span(row):>17s}" for row in a))
ax.set(xscale="log", xlabel="training step", ylabel="MSE on the test set (3 classes)")
ax.set_title("one token for the whole image: median and range of 3 seeds", fontsize=10)
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
fmt_axes(fig)
plt.tight_layout()
plt.show()
post = np.array([p["mse"] for p in R["gridPost"]])
print(f"\n7 × 7 grid, {K} codes: clustering on top of the continuous autoencoder {span(post)} · reference VQ-VAE "
      f"(quantized during training) {span(seeds('ref'))} · continuous, no quantization {span(seeds('ae'))}")  # fmt: skip
print(
    f"clustering on top / quantized during training: {num(np.median(post) / np.median(seeds('ref')), 1)} times the MSE"
)
last = t_steps[-1]
print(f"a first-level token falls in its most frequent class (step {num(last, 0)}, 3 seeds): plain + clustering "
      f"{span(toy_seeds('ae', 'purity', None, last), 1, pct)}, quantized during training "
      f"{span(toy_seeds('vq', 'purity', None, last), 1, pct)}")  # fmt: skip

**Depends on training time, then holds.** Early on, clustering on top wins: the version quantized during training
has to learn the network and the codebook at the same time, and is slower. At step 4,000 the two overlap; at step
10,000 the version quantized during training wins on all 3 seeds. The striking part is the solid orange line: the
longer the plain autoencoder trains, the *worse* its MSE after a codebook is put on top, even though its unquantized
MSE keeps falling. Nothing makes it keep a code space that is easy to quantize: it is free to use small differences
between points to encode detail, and 32 codes can't keep those differences. On the 7 × 7 grid of section 6 (the
lines printed under the figure), the gap is even larger: clustering on top of the continuous autoencoder gives 1.6
times the MSE of the reference VQ-VAE.

**Which one does the second level help more?** On the plain autoencoder, the second level cuts the MSE sharply
(dotted orange). On the version quantized during training, it only brings the MSE down to that version's own ceiling
(dotted green on top of dashed green), and this ceiling is higher than the plain autoencoder's: the commitment loss
keeps $z_e$ close to the codes, so the residual is small and the decoder never learned to use it. The article also
finds the difference between one and two levels rather small. That is why real tokenizers (SoundStream, Mimi, or CAT
in Topic 05) train *all* RVQ levels together, and randomly drop the later levels during training (quantizer
dropout), instead of putting extra levels on top of a model that has finished training.

## 9 · Testing six claims

Each claim below is a sentence from the paper or from a widely cited source. The script trained every configuration
for 3,000 steps with 3 seeds; we read the results on the test set and conclude *holds*, *does not hold* or *depends*,
as Topics 02–05 do. When two seed ranges overlap, the difference is within the seed spread.

| | Claim | Source |
|---|---|---|
| A | Results don't change for β from 0.1 to 2.0 | the paper, section 3.2 |
| B | Without the commitment loss, $z_e$ grows arbitrarily | the paper, section 3.2 |
| C | The EMA version trains faster than the codebook-loss version | the docstring of `VectorQuantizerEMA` in Sonnet (DeepMind) |
| D | Quantization reconstructs worse than a continuous code | the paper, section 4.1 (4.67 against 4.51 bits/dim on CIFAR-10); Rombach et al. (2022), Table 8 |
| E | Bigger codebooks reconstruct better, but without tricks they aren't fully used | Dhariwal et al. (2020), section 3.1; Yu et al. (2021) |
| F | FSQ uses almost the whole codebook without tricks; for small codebooks, VQ is slightly better | Mentzer et al. (2023) |

The ‖z_e‖ column in the tables is the mean length of the encoder's vectors on the test set.

### A and B · The commitment weight β

In [ ]:
LABEL = {
    "ref": "reference (K 128, β 0.25)",
    "ema": "EMA instead of the codebook loss",
    "ema_randn": "EMA, N(0, 1), no restarts",
    "ae": "continuous autoencoder",
    **{f"K{k}": f"K = {num(k, 0)}" for k in (16, 32, 64, 256, 512, 1024)},
    "K512_norestart": "K = 512, no restarts",
    "K1024_norestart": "K = 1,024, no restarts",
    **{f"beta{b:g}": f"β = {num(b, 2 if b == 0.25 else 1)}" for b in (0.0, 0.1, 1.0, 2.0)},
    "fsq128": "FSQ (8, 4, 4) = 128 codes",
    "fsq1000": "FSQ (8, 5, 5, 5) = 1,000 codes",
}


def table(names):
    print(f"{'configuration':36s}{'test MSE':>18s}{'codes used':>14s}{'perplexity':>14s}{'‖z_e‖':>16s}")
    for n in names:
        has = "used" in R["runs"][n][0]["test"]
        used = span(seeds(n, "used"), 0) if has else "—"
        ppl = span(seeds(n, "ppl"), 1) if has else "—"
        z = seeds(n, "zNorm")
        print(f"{LABEL[n]:36s}{span(seeds(n)):>18s}{used:>14s}{ppl:>14s}{span(z, 2 if z.max() < 100 else 0):>16s}")


table(["ref", "beta0.1", "beta1", "beta2"])
for n in ("ref", "beta0"):
    end = [run["checks"][str(STEPS)] for run in R["runs"][n]]
    print(f"{LABEL[n]:26s} on the validation set (float32, step {num(STEPS, 0)}): MSE {span([e['mse'] for e in end])}, "
          f"‖z_e‖ {span([e['zNorm'] for e in end], 2 if n == 'ref' else 0)}")  # fmt: skip
bad = [run["seed"] for run in R["runs"]["beta0"] if run["test"]["mse"] is None]
print(f"β = 0, seed {', '.join(map(str, bad))}: the codebook has a value above 65,504, the largest number float16 can "
      "hold, so its float16 copy is broken")  # fmt: skip

**A · mostly holds.** β from 0.1 to 1 gives an MSE within the seed spread of β = 0.25; β = 2 is slightly worse, but
consistently so on all 3 seeds. What β really changes is the length of $z_e$: the stronger the commitment, the more
the encoder is pulled in close to the codes, and the shorter $z_e$ is. The paper also notes that the right value of β
depends on the scale of the reconstruction loss: here, a mean MSE over pixels in [0, 1].

**B · holds, very clearly.** Dropping the commitment entirely (β = 0), the length of $z_e$ grows from about 2.5 to
tens of thousands, and the MSE gets three to four times worse. Nothing keeps the encoder in place: the codebook loss
makes the codes chase $z_e$, but the encoder runs faster, just as section 3.2 of the paper describes. The codes grow
so large that for one seed the float16 copy the script stores can't hold them, so the β = 0 row is read on the
validation set, from the numbers the script recorded while training in float32.

### C · EMA instead of the codebook loss

In [ ]:
ck = [s for s in steps_axis if s <= STEPS]
print("validation MSE (median of 3 seeds) by step")
print(f"{'step':>8s}" + "".join(f"{s:>9d}" for s in ck))
for n in ("ref", "ema"):
    row = [np.median([run["checks"][str(s)]["mse"] for run in R["runs"][n]]) for s in ck]
    print(f"{n:>8s}" + "".join(f"{num(v, 4):>9s}" for v in row))
table(["ref", "ema", "ema_randn"])
u = {s: span([run["checks"][str(s)]["used"] for run in R["runs"]["ema_randn"]], 0) for s in (1000, 2000)}
print(f"EMA, N(0, 1): {u[1000]} codes at step 1,000, {u[2000]} codes at step 2,000")
t_shrink = math.log(1 / 1e-5) / -math.log(0.99)
print(
    f"an unchosen code starts shrinking toward the origin when 0.99ᵗ < ε = 10⁻⁵, that is after t ≈ {num(t_shrink, 0)} steps"
)

**C · does not hold here.** With the same initialization from data and the same restarts, the two update rules
follow the same curve at every step, and the final results are within the seed spread. EMA differs in a place nobody
advertises: with N(0, 1) and *no* restarts, the EMA version revives a few dozen codes on its own, while the
codebook-loss version doesn't (the table in section 5: 3–6 codes at step 3,000). The reason lies in Sonnet's Laplace
smoothing: for a code that nobody chooses, both the sum `ema_sum` and the counter `count` shrink as $0.99^t$, but the
denominator gets ε = 10⁻⁵ added. Once $0.99^t$ drops below ε, the code gradually shrinks toward the origin, right
where the $z_e$ lie, and gets chosen again; the number of codes used jumps between steps 1,000 and 2,000, just when
the cell above predicts. This is a property of this implementation (the counter starts at 1); other implementations
of the same formula treat dead codes differently.

### D · Quantization and reconstruction quality

People often say VQ-VAE images are "sharper" than a VAE's because the code is discrete. Let's test it directly.

### 🤔 Predict first
Same network, same 7 × 7 × 16 grid, same training; one side quantizes each cell to 1 of 128 codes, the other keeps
the 16 real numbers. Which one reconstructs the test images with the lower MSE?

In [ ]:
table(["ae", "ref", "K1024"])
print(
    f"continuous autoencoder vs reference VQ-VAE: {num(np.median(seeds('ref')) / np.median(seeds('ae')), 2)} times lower MSE"
)

**D · holds.** The continuous autoencoder reconstructs much better: an MSE 2.38 times lower than the reference
VQ-VAE's, and still lower than the VQ-VAE with 1,024 codes. Quantization is an *added* bottleneck; it can only lose
information. The paper saw the same (4.67 bits/dim against 4.51 for a continuous VAE on CIFAR-10), and Latent
Diffusion found VQ-regularized autoencoders reconstruct slightly worse than KL-regularized ones. VQGAN's images are
sharp thanks to its perceptual and GAN losses (Esser et al., 2021), not thanks to quantization. What VQ buys is
something else: 343 bits instead of 25,088, and a sequence of tokens a Transformer can learn (section 10).

### E · Codebook size

The figure below puts every configuration on the same axes: the bits of one image (49 · log₂ number of codes) and
the MSE on the test set. The FSQ points are explained in F right after.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ks = (16, 32, 64, 128, 256, 512, 1024)
kname = {k: "ref" if k == 128 else f"K{k}" for k in ks}
for names, c, mk, dx, label in (
    ([kname[k] for k in ks], GREEN, "o", 0, "VQ, with restarts"),
    (["K512_norestart", "K1024_norestart"], ORANGE, "s", 5, "VQ, no restarts"),
    (["fsq128", "fsq1000"], BLUE, "D", -5, "FSQ (8, 4, 4) and (8, 5, 5, 5)"),
):
    size = [np.prod(R["grid"][n]["levels"]) if "fsq" in n else R["runs"][n][0]["cfg"]["K"] for n in names]
    bits = np.array([49 * math.log2(k) for k in size]) + dx  # shifted a few bits sideways so the points don't overlap
    med = np.array([np.median(seeds(n)) for n in names])
    err = [med - [seeds(n).min() for n in names], [seeds(n).max() for n in names] - med]
    ax.errorbar(bits, med, yerr=err, color=c, marker=mk, ls="-" if c == GREEN else "none", capsize=3, label=label)
ax.axhline(np.median(seeds("ae")), color=GREY, ls="--", label=f"continuous autoencoder ({num(bits_float, 0)} bits)")
ax.set_xticks([49 * math.log2(k) for k in ks], [f"{num(49 * math.log2(k), 0)}\nK {num(k, 0)}" for k in ks])
ax.set(xlabel="bits per image (49 · log₂ number of codes)", ylabel="MSE on the test set", ylim=(0, None))
ax.legend(fontsize=8)
fmt_axes(fig)
plt.tight_layout()
plt.show()
table(["K16", "K64", "ref", "K256", "K512", "K1024", "K512_norestart", "K1024_norestart"])

**E · holds.** With restarts, every K uses nearly its whole codebook, and the MSE falls steadily as the bits grow
(green line): doubling K adds 49 bits, and the MSE drops a little each time. Even 1,024 codes (490 bits) are still
far from the continuous autoencoder (grey line). Without restarts (orange squares), K = 512 and K = 1,024 waste most of
their codebook, and their MSE is worse than with restarts at the same K, on a par with much smaller codebooks:
K = 512 matches K = 128–256, K = 1,024 matches K = 64–128. Their seed ranges are also much wider: how many codes
survive is a matter of luck.

### F · FSQ: no codebook at all

FSQ (*finite scalar quantization*, Mentzer et al., 2023) replaces the learned codebook with a fixed grid. The encoder
outputs only a few numbers per cell (here 3 or 4 instead of 16); each one is bounded with tanh and rounded to one of
$L_i$ levels, with straight-through for the rounding. Levels (8, 4, 4) give 8 · 4 · 4 = 128 combinations, exactly the
reference model's K; (8, 5, 5, 5) give 1,000. There is no codebook loss, no commitment, and no code can "die" in the
sense of section 5, because every point of the grid always lies within the region $z_e$ can reach. NVIDIA's Cosmos
tokenizer (2025) uses FSQ.

In [ ]:
table(["ref", "fsq128", "K1024", "K1024_norestart", "fsq1000"])
for run in R["runs"]["fsq1000"]:
    u = [int(run["checks"][str(s)]["used"]) for s in (1000, 2000, STEPS)]
    print(
        f"FSQ 1,000 codes, seed {run['seed']}: {u[0]} → {u[1]} → {u[2]} codes used at step 1,000 → 2,000 → {num(STEPS, 0)}"
    )

**F · depends.** With 128 codes, FSQ uses all 128 without any trick, and VQ beats it slightly on MSE, just as the
paper says for small codebooks. With 1,000 codes, FSQ uses only a third to two thirds of the codes after 3,000 steps,
not nearly 100% as the paper reports; it is on a par with VQ *without* restarts, while VQ with restarts uses them all
and reconstructs better. The paper measures much larger models trained many times longer (MaskGIT, UViM); at this
scale, FSQ's advantage of "using the whole codebook" doesn't show yet, although for two of the three seeds the number
of codes it uses is still rising at the last step.

## 10 · New images: a prior over the tokens

The decoder only turns tokens into images; to *generate* new images, we need to know which token grids are
plausible, that is a distribution $p(\text{token}_1, \dots, \text{token}_{49})$. During training, VQ-VAE treats this
distribution as uniform: every grid is equally likely. Then the KL term of Lesson 19's ELBO is a constant, log K per
token, and is dropped from the loss (section 3.1 of the paper). Only *after* training does the paper learn an
autoregressive prior over the tokens (a PixelCNN, section 3.3).

We use a small causal Transformer like Lesson 17's: it reads the 7 × 7 grid row by row, like a 49-word sentence, and
predicts each token from the tokens before it:

$$p(\text{token}_1, \dots, \text{token}_{49}) = \prod_{t=1}^{49} p(\text{token}_t \mid \text{token}_1, \dots,
\text{token}_{t-1}) .$$

The script trained it on the tokens of the 55,000 training images, with η chosen on the validation set. The measure
is **bits per image**: $-\log_2 p(\text{the image's tokens})$, averaged over the test images. The better the prior
understands clothes, the fewer bits; it is also the average length if we used the prior to compress images
(arithmetic coding). We compare it with three simple priors, all smoothed by adding 1 to every count: uniform
(log₂ 128 = 7 bits per token), the overall frequency of each code, and the frequency of each code *at each position*.
The cell below recomputes all four on the first 2,000 test images and compares them with the script.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.qkv, self.proj = nn.LayerNorm(d), nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.ln2, self.fc, self.out = nn.LayerNorm(d), nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.heads = heads

    def forward(self, h):
        B, T, d = h.shape
        q, k, v = self.qkv(self.ln1(h)).view(B, T, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # each position sees only the tokens before it
        h = h + self.proj(a.transpose(1, 2).reshape(B, T, d))
        return h + self.out(F.gelu(self.fc(self.ln2(h))))


class Prior(nn.Module):
    """p(token₁ … token₄₉) = Π p(tokenₜ | the tokens before), reading the 7 × 7 grid by rows; token K is "start"."""

    def __init__(self, K=128, d=128, layers=4, heads=4, T=GRID * GRID):
        super().__init__()
        self.tok, self.pos = nn.Embedding(K + 1, d), nn.Parameter(torch.randn(T, d) * 0.02)
        self.blocks = nn.ModuleList(Block(d, heads) for _ in range(layers))
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, K)
        self.K = K

    def forward(self, c):
        """c: (B, t) tokens so far → logits (B, t, K) for the next token at each position (position 0 reads "start")."""
        inp = torch.cat([torch.full_like(c[:, :1], self.K), c[:, :-1]], 1)
        h = self.tok(inp) + self.pos[: c.shape[1]]
        for b in self.blocks:
            h = b(h)
        return self.head(self.ln(h))

    def nll_bits(self, c, chunk=1000):
        """Mean −log₂ p(image) over the sequences c (n, 49): the bits needed to encode an image with this prior."""
        with torch.no_grad():
            tot = sum(
                float(F.cross_entropy(self(c[i : i + chunk]).transpose(1, 2), c[i : i + chunk], reduction="sum"))
                for i in range(0, len(c), chunk)
            )
        return tot / len(c) / math.log(2)


def baseline_bits(Ctr, Cte, K):
    """−log₂ p(image) on the test set for three simple priors: uniform, overall code frequency, per-position frequency."""
    n_te = len(Cte)
    flat_te = Cte.reshape(n_te, -1)
    uni = torch.bincount(Ctr.reshape(-1), minlength=K).double() + 1
    uni /= uni.sum()
    pos = torch.stack([torch.bincount(Ctr.reshape(len(Ctr), -1)[:, j], minlength=K) for j in range(GRID * GRID)])
    pos = (pos.double() + 1) / (pos.sum(1, keepdim=True) + K)
    return (
        {
            "uniform": GRID * GRID * math.log2(K),
            "unigram": float(-torch.log2(uni[flat_te]).sum(1).mean()),
            "position": float(-torch.log2(pos[torch.arange(GRID * GRID), flat_te]).sum(1).mean()),
        },
        uni,
        pos,
    )


pc = W["cfg"]["prior"]
prior = Prior(K, pc["d"], pc["layers"])
prior.load_state_dict(unpack(W["prior"]))
prior.eval()
print(
    f"prior: {pc['layers']} Transformer blocks, width {pc['d']}, {num(sum(p.numel() for p in prior.parameters()), 0)} parameters"
)

C_tr = codes(vqm, X)
N_LIVE = R["prior"]["cfg"]["nllLive"]
base, p_uni, p_pos = baseline_bits(C_tr, C_te[:N_LIVE], K)
got = {**base, "transformer": prior.nll_bits(C_te[:N_LIVE].reshape(N_LIVE, -1))}
want = R["prior"]["seeds"][0]["testLive"]
for k in got:
    assert abs(got[k] - want[k]) < 0.05 * (1 if k != "transformer" else 4), f"{k}: differs from the script"
NAMES = {"uniform": "uniform (every code alike)", "unigram": "overall code frequency", "position": "frequency per position",
         "transformer": "autoregressive Transformer"}  # fmt: skip
print(f"\n{'prior':28s}{f'bits/image, first {num(N_LIVE, 0)}':>26s}{'bits/image, whole test set (3 seeds)':>40s}")
for k, name in NAMES.items():
    full = [s["test"][k] for s in R["prior"]["seeds"]]
    print(f"{name:28s}{num(got[k], 1):>26s}{span(full, 1):>40s}")
print(
    f"\nsaved over the uniform prior: overall frequency {num(got['uniform'] - got['unigram'], 1)} bits, per position "
    f"{num(got['uniform'] - got['position'], 1)} bits, Transformer {num(got['uniform'] - got['transformer'], 1)} bits"
)
print(
    f"original image {num(bits_raw, 0)} bits / Transformer {num(got['transformer'], 1)} bits = {num(bits_raw / got['transformer'], 1)} times"
)

The uniform prior costs exactly 343 bits, 49 · log₂ 128: it knows nothing. Knowing the overall frequency of each code
saves only 16.5 bits; knowing the frequency at each position (corners are almost always background, the middle is
fabric) saves 46.5 bits. The Transformer needs only about half the bits of the uniform prior: once it has seen the
earlier tokens, the next one is much easier to guess. In terms of compression: the original image's 6,272 bits
become about 170, 36.6 times smaller, but that is *lossy* compression: the rebuilt image is blurry, as in section 6.

### 🤔 Predict first
The "per position" prior knows, for example, that corners are almost always a background code and the middle is
often a fabric code. Draw each token *independently* from exactly those frequencies at its position, then decode:
does the result look like a garment?

In [ ]:
@torch.no_grad()
def sample(prior, n, g, temp=1.0, prefix=None):
    """Draw tokens one by one in row order; prefix (n, t) holds given first tokens (for completing an image)."""
    c = torch.zeros(n, 0, dtype=torch.long) if prefix is None else prefix.clone()
    while c.shape[1] < GRID * GRID:
        logits = prior(torch.cat([c, torch.zeros(n, 1, dtype=torch.long)], 1))[:, -1] / temp
        c = torch.cat([c, torch.multinomial(F.softmax(logits, -1), 1, generator=g)], 1)
    return c.view(n, GRID, GRID)


g = torch.Generator().manual_seed(0)
n = 12
draws = {
    "uniform": torch.randint(K, (n, GRID, GRID), generator=g),
    "per position": torch.stack([torch.multinomial(p_pos[j].float(), n, True, generator=g) for j in range(49)], 1),
    "Transformer": sample(prior, n, g),
}
with torch.no_grad():
    rows = {k: vqm.decode(c.view(n, GRID, GRID)) for k, c in draws.items()}
    rows["real images (rebuilt)"] = vqm(T[:n])[0]
fig, axes = plt.subplots(4, 1, figsize=(10, 4.4))
for ax, (name, imgs) in zip(axes, rows.items(), strict=True):
    show_row(ax, imgs, name)
plt.tight_layout()
plt.show()

Each row is 12 samples. **Uniform:** not clothes at all, just patchwork. **Per position:** almost the same blob every
time, an "average garment" with sleeves and trouser legs at once, because each cell is drawn on its own and no cell
knows which item its neighbor belongs to. The frequency at each position is right, but the coordination between
positions is lost. **Transformer:** trousers, tops, bags and shoes you can recognize, although blurrier and more
distorted than the rebuilt real images in the last row. The decoder is unchanged; it is the prior that turns the
tokenizer into a generative model.

### Completing an image

Because the prior reads the tokens row by row, we can give it the first few rows and let it write the rest, like
giving a language model the beginning of a sentence. Each row of the figure below is a test image: the original, the
image with only its first 2 rows of tokens (14 of the 49 tokens; the rest replaced by a background code), then four
different completions.

In [ ]:
KEEP = 2 * GRID
g = torch.Generator().manual_seed(1)
fig, axes = plt.subplots(len(pick[:6]), 1, figsize=(6.5, 6.4))
for ax, i in zip(axes, pick[:6].tolist(), strict=True):
    c = C_te[i].reshape(1, -1)
    cut = c.clone()
    cut[:, KEEP:] = bg
    done = sample(prior, 4, g, prefix=c[:, :KEEP].repeat(4, 1))
    with torch.no_grad():
        imgs = torch.cat([T[i : i + 1], vqm.decode(cut.view(1, GRID, GRID)), vqm.decode(done)])
    show_row(ax, imgs, CLASSES[int(y_te[i])])
axes[0].set_title("original · first 2 rows of tokens · four completions", fontsize=9)
plt.tight_layout()
plt.show()

The first two rows of tokens are only the top 8 rows of pixels, and the prior completes them with whatever fits. The
top of a sleeveless top looks like the waistband of a trouser or the shoulders of a dress, so the completions can
turn into different items; the broad shoulders of the pullover and the coat give garments with sleeves. The sandal is
so low that its first two rows are all background: the prior only knows "the top is empty", and completes it as
footwear, the kind of item with an empty top. This is exactly how a language model continues a sentence, except that
the "sentence" here is an image.

## 11 · Try it yourself

### Sampling temperature

Divide the logits by a temperature T before the softmax (as in Lesson 17): T < 1 concentrates the probability on the
common tokens, T > 1 spreads it out. Change `TEMPERATURES` and run again.

In [ ]:
TEMPERATURES = (0.6, 1.0, 1.4)
g = torch.Generator().manual_seed(2)
fig, axes = plt.subplots(len(TEMPERATURES), 1, figsize=(10, 1.2 * len(TEMPERATURES) + 0.4))
for ax, t in zip(np.atleast_1d(axes), TEMPERATURES, strict=True):
    c = sample(prior, 12, g, temp=t)
    with torch.no_grad():
        show_row(ax, vqm.decode(c), f"T = {num(t, 1)}")
    print(
        f"T = {num(t, 1)}: {len(torch.unique(c))} different codes in 12 images, {pct(float(is_bg[c].float().mean()))} of the cells background codes"
    )
plt.tight_layout()
plt.show()

### Training again with another configuration

The cell below trains again for a few hundred steps with a configuration you choose, then compares it with the
script's full run at the same step (if that configuration is in the grid). Try `init="uniform"`, `restart=True` with
`init="randn"`, `K=512`, `beta=0` or `update="ema"`.

In [ ]:
CONFIG = {**REF, "init": "randn", "restart": False}
TRY_STEPS, SEED = 200, 0
m_try = train(CONFIG, SEED, TRY_STEPS)
res = metrics(m_try, V)
print(
    f"{TRY_STEPS} steps: validation MSE {num(res['mse'], 5)}, {res.get('used', '—')} codes used, {m_try.q.restarts} codes restarted"
)
for name, run in R["runs"].items():
    if run[SEED]["cfg"] == CONFIG and str(TRY_STEPS) in run[SEED]["checks"]:
        s = run[SEED]["checks"][str(TRY_STEPS)]
        print(f"script ({name}, same step): MSE {num(s['mse'], 5)}, {int(s.get('used', 0))} codes; after {num(STEPS, 0)} steps: "
              f"{span(seeds(name), 4)} MSE, {span(seeds(name, 'used'), 0)} codes (3 seeds)")  # fmt: skip

### Editing tokens

Tokens are a discrete "drawing": change a few tokens and you change a region of the image. The cell below takes the
trouser from section 6 and replaces its middle column of tokens (column 4, counting from 1) with a background code,
to see what the decoder draws into the gap. Try another row, another code.

In [ ]:
i = int(pick[1])
c = C_te[i].clone()
c[:, 3] = bg
with torch.no_grad():
    imgs = torch.cat([T[i : i + 1], vqm.decode(C_te[i : i + 1]), vqm.decode(c[None])])
fig, ax = plt.subplots(figsize=(4, 1.6))
show_row(ax, imgs)
ax.set_title("original · rebuilt · middle token column set to background", fontsize=9)
plt.show()

## 12 · Limits and what comes next

- **Small.** 28 × 28 grayscale images, a network of 31,377 parameters, 3,000 steps; real models use color images of
  256 × 256 and up, networks of tens of millions of parameters and hundreds of thousands of steps. The conclusions
  here are about the *mechanism*, not about absolute numbers.
- **MSE only.** When you're unsure where a detail goes, guessing the average of the possibilities gives the lowest
  MSE, so the squared loss favors blurry images (Lesson 19). VQGAN adds a perceptual loss and a GAN discriminator on
  image patches (Lesson 20), and that is what makes its images sharp.
- **One level, one grid.** VQ-VAE-2 uses two grids (32 × 32 and 64 × 64 for 256 × 256 images); audio tokenizers
  stack several codebooks on top of each other on the residual (RVQ, Topic 05), and train all levels together
  instead of putting extra levels on top as in section 8.
- **A small prior that reads row by row.** MaskGIT replaces the autoregressive prior with a bidirectional Transformer
  that guesses many masked tokens at once, generating an image in 8 steps instead of 256.
- **3 seeds, one η.** η was chosen for the reference configuration and then shared; another configuration (FSQ,
  β = 0) might do better at its own η.

What we built here is the backbone of many large systems. DALL·E (2021) compresses a 256 × 256 image into 32 × 32
tokens (8,192 codes, but learned with Gumbel-softmax instead of straight-through). Parti generates ViT-VQGAN tokens
from a caption; Chameleon puts image tokens in the same sequence as text tokens: its tokenizer turns a 512 × 512
image into 1,024 tokens from a codebook of 8,192 codes. Latent Diffusion (Lesson 21) uses the same kind of
autoencoder, but diffuses over continuous codes. And every audio tokenizer of Topic 05 is a VQ-VAE with several
levels of codes.

## Summary

- **Image tokenization:** a CNN encoder turns a 28 × 28 image into a 7 × 7 grid of vectors; each vector becomes the
  index of its nearest code among 128. An image becomes 49 tokens, 343 bits, 18.3 times smaller than the original.
- **Straight-through** copies the gradient across the nearest-code step, so the codebook gets no gradient from the
  reconstruction. The two added terms are not symmetric: the codebook loss trains the codebook, the commitment loss
  (β = 0.25) trains the encoder.
- **Dead codebooks:** only chosen codes learn. An N(0, 1) initialization leaves only a few usable codes and never
  recovers; initializing from data plus dead-code restarts (Jukebox) uses all 128 codes.
- **Codes are local patterns**, not kinds of garment; many codes are copies of the black background, so "all 128
  codes in use" doesn't mean 128 different things.
- **One token for the whole image** (Kyutai's article): a token nearly names the garment (more than 90% of the time
  one class), but 32 codes can rebuild only 32 images. Quantizing during training beats clustering on top when trained
  long enough; an RVQ level put on top only helps when the autoencoder hasn't been pulled onto its codes by the
  commitment loss.
- **The claims:** A mostly holds (β 0.1–1 within the spread, β 2 slightly worse); B holds (β = 0 makes $z_e$ more
  than ten thousand times longer); C does not hold here (EMA is on a par with the codebook loss, but partly rescues a
  bad initialization on its own); D holds (the continuous autoencoder has a 2.38 times lower MSE); E holds (bigger K
  is better, but needs restarts); F depends (FSQ uses all 128 codes, but only part of 1,000).
- **Generating images** needs a prior over the tokens. A small Transformer (119,808 parameters) encodes a test image
  in 168.4–172.7 bits, against 343 for the uniform prior and 296.8–298.0 for the per-position prior; its samples are
  recognizable, while drawing each position independently gives only a blur. It can also complete an image from its
  first few rows of tokens.

## Review questions

1. Why is the gradient of the reconstruction loss with respect to the codebook exactly 0? If you drop the codebook
   loss but keep the commitment loss, can the codebook learn?
2. A 16-dimensional N(0, 1) codebook leaves only a few codes in use, and training can't fix it. Explain this with the
   lengths of $z_e$ and of the codes. Which two fixes did we use here?
3. A tokenizer for 256 × 256 images uses a 32 × 32 grid and 8,192 codes. How many bits is one image? How many bits is
   the original, at 8 bits per color channel?
4. The reference model uses all 128 codes, but many are copies of the black background. Why do dead-code restarts
   create copies? Does a higher perplexity always mean a better codebook?
5. Drawing each token independently from its frequency at its position gives almost the same blur every time. Why?
   What does the Transformer prior know in addition?
6. The continuous autoencoder reconstructs better than VQ-VAE. Why, then, do systems such as DALL·E, Parti or
   Chameleon still quantize images?
7. In section 8, clustering on top of a plain autoencoder gets *worse* as the autoencoder trains longer. Why? And why
   does an RVQ level put on top help the plain autoencoder more than the version quantized during training?

<details><summary>Hints</summary>

1. The decoder's input is `ze + (zq − ze).detach()`: its value is `zq`, its gradient goes to `ze`; `zq` appears only
   inside the `.detach()`, so there is no path from the reconstruction loss back to the codebook. The commitment loss
   $\lVert z_e - \mathrm{sg}[e]\rVert^2$ also stops the gradient at $e$. Without the codebook loss, the codebook stays
   at its initial values (unless you use the EMA version, which updates the codes without gradients).
2. At initialization, $z_e$ has length about 0.4 and huddles near the origin, while the N(0, 1) codes have length
   about 4. The nearest code of most $z_e$ is the same one or two codes that happen to lie closest to the origin. The
   other codes are never chosen, so they get no gradient from the codebook loss, so they never move toward $z_e$. The
   fixes: use the $z_e$ of the first batch as the codebook, and replace codes nobody uses with a random $z_e$ of the
   current batch.
3. log₂ 8,192 = 13 bits, so 32 · 32 · 13 = 13,312 bits. The original: 256 · 256 · 3 · 8 = 1,572,864 bits, 118 times
   more. (That is DALL·E's setup, although DALL·E learns its codebook with Gumbel-softmax.)
4. A dead code is reset to a random $z_e$ of the batch, and half of all pixels are black background, so a great many
   $z_e$ are "background". New codes therefore tend to land in the background region and split up background cells
   that need only one code. Perplexity measures how evenly the codes are used, not whether they differ; Huh et al.
   (2023) also warn that a high perplexity can just be redundancy.
5. Each position is drawn on its own, so the token in one cell doesn't know whether the neighboring cell is a shirt or
   a trouser: the result is a patchwork average of every item. The Transformer prior reads the earlier tokens, so if
   the first two rows are a collar, it knows the next rows must be the body of a shirt.
6. Because tokens give two things a continuous code doesn't: few bits, and a sequence of integers that a Transformer
   learns exactly the way it learns text (predicting the next token with cross-entropy), sharing one model for text
   and images. The quality lost in quantization is made up for by stronger decoders and by perceptual and GAN losses.
7. The plain autoencoder is rewarded only for reconstructing well from 2 real numbers, so the longer it trains, the
   more it uses small differences between points to encode detail; 32 codes put on afterwards erase those
   differences. The version quantized during training has its $z_e$ pulled close to the codes by the commitment loss,
   and its decoder has only ever seen those 32 points: the residual is small, and the decoder never learned to use
   it. An extra level only brings it to its own ceiling. For many levels to help, they must be trained together with
   the network, as in SoundStream or CAT in Topic 05.
</details>

## Further reading

- van den Oord, Vinyals & Kavukcuoglu (2017), *Neural Discrete Representation Learning* (VQ-VAE):
  [arXiv:1711.00937](https://arxiv.org/abs/1711.00937).
- Razavi, van den Oord & Vinyals (2019), *Generating Diverse High-Fidelity Images with VQ-VAE-2*:
  [arXiv:1906.00446](https://arxiv.org/abs/1906.00446).
- Dhariwal et al. (2020), *Jukebox: A Generative Model for Music* (dead-code restarts):
  [arXiv:2005.00341](https://arxiv.org/abs/2005.00341).
- Esser, Rombach & Ommer (2021), *Taming Transformers for High-Resolution Image Synthesis* (VQGAN):
  [arXiv:2012.09841](https://arxiv.org/abs/2012.09841).
- Yu et al. (2021), *Vector-quantized Image Modeling with Improved VQGAN*:
  [arXiv:2110.04627](https://arxiv.org/abs/2110.04627).
- Chang et al. (2022), *MaskGIT: Masked Generative Image Transformer*:
  [arXiv:2202.04200](https://arxiv.org/abs/2202.04200).
- Rombach et al. (2022), *High-Resolution Image Synthesis with Latent Diffusion Models*:
  [arXiv:2112.10752](https://arxiv.org/abs/2112.10752).
- Huh et al. (2023), *Straightening Out the Straight-Through Estimator: Overcoming Optimization Challenges in Vector
  Quantized Networks*: [arXiv:2305.08842](https://arxiv.org/abs/2305.08842).
- Mentzer et al. (2023), *Finite Scalar Quantization: VQ-VAE Made Simple*:
  [arXiv:2309.15505](https://arxiv.org/abs/2309.15505).
- Fifty et al. (2024), *Restructuring Vector Quantization with the Rotation Trick*:
  [arXiv:2410.06424](https://arxiv.org/abs/2410.06424).
- Chameleon Team (2024), *Chameleon: Mixed-Modal Early-Fusion Foundation Models*:
  [arXiv:2405.09818](https://arxiv.org/abs/2405.09818).
- NVIDIA (2025), *Cosmos World Foundation Model Platform for Physical AI* (an FSQ tokenizer):
  [arXiv:2501.03575](https://arxiv.org/abs/2501.03575).
- DeepMind's original code: [`sonnet/src/nets/vqvae.py`](https://github.com/google-deepmind/sonnet/blob/v2/sonnet/src/nets/vqvae.py)
  and [`vqvae_example.ipynb`](https://github.com/google-deepmind/sonnet/blob/v2/examples/vqvae_example.ipynb).
- Volhejn (2025), *Neural audio codecs: how to get audio into LLMs* (Kyutai): [kyutai.org/codec-explainer](https://kyutai.org/codec-explainer/),
  code in [`kyutai-labs/nanoGPTaudio`](https://github.com/kyutai-labs/nanoGPTaudio). Section 8 rebuilds the article's
  Fashion-MNIST part; the rest goes on to RVQ for audio, Mimi and semantic tokens.
- Zeghidour et al. (2021), *SoundStream: An End-to-End Neural Audio Codec* (RVQ for audio):
  [arXiv:2107.03312](https://arxiv.org/abs/2107.03312).
- Xiao, Rasul & Vollgraf (2017), *Fashion-MNIST*: [arXiv:1708.07747](https://arxiv.org/abs/1708.07747).

**Back to:** [Lesson 19 · Autoencoders & VAEs](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/autoencoder-vae/) ·
[Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/) ·
[Topic 05 · Audio tokenization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-hoa-am-thanh/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)